# Question Answering - Seq2Seq Architectures
This notebook trains RNN, LSTM, GRU, and BiGRU encoder-decoder models on Arabic QA generation.

The workflow compares embeddings across all datasets (raw plus all preprocessed variants):
- Bag of Words
- TF-IDF
- Word2Vec Skip-gram
- Word2Vec CBOW
- fastText
- BERT sentence embeddings
- GPT sentence embeddings

Metrics: BLEU and ROUGE-L.

In [2]:
import os
import re
import gc
import json
import time
import math
import glob
import random
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple

print('Finished importing standard libraries.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import joblib

print('Finished importing third-party libraries.')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print('Finished importing PyTorch.')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

print('Finished importing NLP libraries.')

from gensim.models import Word2Vec, FastText
from transformers import AutoTokenizer, AutoModel

print('Finished importing embedding libraries.')

warnings.filterwarnings('ignore')
print('[INFO] Imports completed.')

Finished importing standard libraries.
Finished importing third-party libraries.
Finished importing PyTorch.
Finished importing NLP libraries.
Finished importing embedding libraries.
[INFO] Imports completed.


In [ ]:
SEED = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 1e-3
EMBED_DIM = 256
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.2
MAX_SRC_LEN = 40
MAX_TGT_LEN = 48
MIN_TOKEN_FREQ = 1
MAX_ROWS_PER_DATASET = None  # Example: 10 for quick smoke test, None for full dataset

DATASET_ROOT = '.'
PREPROCESSED_DIR = os.path.join(DATASET_ROOT, 'preprocessed datasets')
ORIGINAL_DATASET = os.path.join(DATASET_ROOT, 'AAFAQ_Dataset.csv')
OUTPUT_DIR = os.path.join(DATASET_ROOT, 'outputs')
CACHE_DIR = os.path.join(DATASET_ROOT, 'feature_cache', 'qa_seq2seq')
CKPT_DIR = os.path.join(OUTPUT_DIR, 'qa_seq2seq_checkpoints')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

BERT_MODEL_NAME = 'aubmindlab/bert-base-arabertv02'
GPT_MODEL_NAME = 'aubmindlab/aragpt2-base'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'[INFO] Device: {DEVICE}')
print(f'[INFO] Output dir: {OUTPUT_DIR}')
print(f'[INFO] Max rows per dataset: {MAX_ROWS_PER_DATASET}')

[INFO] Device: cuda
[INFO] Output dir: .\outputs


In [4]:
def clean_text(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def build_dataset_registry() -> Dict[str, str]:
    registry = {'Original_raw': ORIGINAL_DATASET}
    if os.path.isdir(PREPROCESSED_DIR):
        for path in sorted(glob.glob(os.path.join(PREPROCESSED_DIR, '*.csv'))):
            name = os.path.splitext(os.path.basename(path))[0]
            registry[name] = path
    return registry

def load_dataset(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    required_cols = {'QuestionText', 'Answer', 'Category'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'Missing columns {missing} in {path}')
    df = df.dropna(subset=['QuestionText', 'Answer']).copy()
    df['QuestionText'] = df['QuestionText'].map(clean_text)
    df['Answer'] = df['Answer'].map(clean_text)
    df = df[(df['QuestionText'].str.len() > 0) & (df['Answer'].str.len() > 0)].reset_index(drop=True)
    return df

DATASET_REGISTRY = build_dataset_registry()
print(f'[INFO] Found {len(DATASET_REGISTRY)} datasets for QA seq2seq experiments.')
for k, v in DATASET_REGISTRY.items():
    print(f'  - {k}: {v}')

qa_datasets = {name: load_dataset(path) for name, path in DATASET_REGISTRY.items()}
pd.DataFrame([{'Dataset': k, 'Rows': len(v)} for k, v in qa_datasets.items()]).sort_values('Rows', ascending=False)

[INFO] Found 7 datasets for QA seq2seq experiments.
  - Original_raw: .\AAFAQ_Dataset.csv
  - pyarabic_aggressive_preprocessed: .\preprocessed datasets\pyarabic_aggressive_preprocessed.csv
  - pyarabic_hamza_only_preprocessed: .\preprocessed datasets\pyarabic_hamza_only_preprocessed.csv
  - pyarabic_hamza_tashkeel_preprocessed: .\preprocessed datasets\pyarabic_hamza_tashkeel_preprocessed.csv
  - pyarabic_punctuation_only_preprocessed: .\preprocessed datasets\pyarabic_punctuation_only_preprocessed.csv
  - pyarabic_tashkeel_tatweel_preprocessed: .\preprocessed datasets\pyarabic_tashkeel_tatweel_preprocessed.csv
  - regex_aggressive_preprocessed: .\preprocessed datasets\regex_aggressive_preprocessed.csv


,Dataset,Rows
0,Original_raw,5009
1,pyarabic_aggressive_preprocessed,5009
2,pyarabic_hamza_only_preprocessed,5009
3,pyarabic_hamza_tashkeel_preprocessed,5009
4,pyarabic_punctuation_only_preprocessed,5009
5,pyarabic_tashkeel_tatweel_preprocessed,5009
6,regex_aggressive_preprocessed,5009


## Data Splits, Vocabulary, and Metrics

In [5]:
def simple_tokenize(text: str) -> List[str]:
    return str(text).split()

class Vocabulary:
    def __init__(self, min_freq: int = 1):
        self.min_freq = min_freq
        self.pad_token = '<pad>'
        self.sos_token = '<sos>'
        self.eos_token = '<eos>'
        self.unk_token = '<unk>'
        self.special_tokens = [self.pad_token, self.sos_token, self.eos_token, self.unk_token]
        self.stoi = {tok: idx for idx, tok in enumerate(self.special_tokens)}
        self.itos = {idx: tok for idx, tok in enumerate(self.special_tokens)}

    def build(self, texts: List[str]):
        counter = {}
        for text in texts:
            for tok in simple_tokenize(text):
                counter[tok] = counter.get(tok, 0) + 1
        for tok, freq in counter.items():
            if freq >= self.min_freq and tok not in self.stoi:
                idx = len(self.stoi)
                self.stoi[tok] = idx
                self.itos[idx] = tok

    def encode(self, text: str, max_len: int) -> List[int]:
        tokens = [self.sos_token] + simple_tokenize(text)[:max_len - 2] + [self.eos_token]
        ids = [self.stoi.get(tok, self.stoi[self.unk_token]) for tok in tokens]
        if len(ids) < max_len:
            ids += [self.stoi[self.pad_token]] * (max_len - len(ids))
        return ids[:max_len]

    def decode(self, ids: List[int]) -> str:
        toks = []
        for idx in ids:
            tok = self.itos.get(int(idx), self.unk_token)
            if tok in {self.pad_token, self.sos_token}:
                continue
            if tok == self.eos_token:
                break
            toks.append(tok)
        return ' '.join(toks)

def compute_bleu(references: List[str], predictions: List[str]) -> float:
    smooth = SmoothingFunction().method4
    scores = []
    for ref, pred in zip(references, predictions):
        ref_tokens = simple_tokenize(ref)
        pred_tokens = simple_tokenize(pred)
        if not pred_tokens:
            scores.append(0.0)
            continue
        scores.append(sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth))
    return float(np.mean(scores)) if scores else 0.0

def compute_rouge_l(references: List[str], predictions: List[str]) -> float:
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    vals = []
    for ref, pred in zip(references, predictions):
        vals.append(scorer.score(ref, pred)['rougeL'].fmeasure)
    return float(np.mean(vals)) if vals else 0.0

def split_dataframe(df: pd.DataFrame):
    train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, shuffle=True)
    train_df, val_df = train_test_split(train_df, test_size=VAL_SIZE, random_state=SEED, shuffle=True)
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

print('[INFO] Vocabulary, metrics, and split utilities ready.')

[INFO] Vocabulary, metrics, and split utilities ready.


## Embedding Builders

In [ ]:
@dataclass
class EmbeddingPack:
    mode: str
    src_train: np.ndarray
    src_val: np.ndarray
    src_test: np.ndarray
    vocab: Vocabulary = None
    pretrained_matrix: np.ndarray = None


def cache_path(dataset_name: str, strategy: str, train_len: int = None, val_len: int = None, test_len: int = None) -> str:
    limit_tag = 'full' if MAX_ROWS_PER_DATASET is None else f'rows{int(MAX_ROWS_PER_DATASET)}'
    shape_tag = ''
    if train_len is not None and val_len is not None and test_len is not None:
        shape_tag = f'_tr{train_len}_va{val_len}_te{test_len}'
    key = f'{dataset_name}_{strategy}_{limit_tag}{shape_tag}'.replace(' ', '_')
    return os.path.join(CACHE_DIR, f'{key}.joblib')


def mean_pool_transformer_embeddings(texts: List[str], model_name: str, batch_size: int = 16, max_len: int = 64) -> np.ndarray:
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModel.from_pretrained(model_name).to(DEVICE)
    model.eval()
    outputs = []
    for i in tqdm(range(0, len(texts), batch_size), desc=f'Embedding {model_name}'):
        batch = texts[i:i + batch_size]
        enc = tok(batch, return_tensors='pt', truncation=True, padding=True, max_length=max_len)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            h = model(**enc).last_hidden_state
        mask = enc['attention_mask'].unsqueeze(-1)
        pooled = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        outputs.append(pooled.detach().cpu().numpy())
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return np.vstack(outputs)


def build_dense_pack(strategy: str, dataset_name: str, train_q: List[str], val_q: List[str], test_q: List[str]) -> EmbeddingPack:
    cp = cache_path(dataset_name, strategy, len(train_q), len(val_q), len(test_q))
    if os.path.exists(cp):
        print(f'[CACHE] Loading {strategy} for {dataset_name}')
        pack = joblib.load(cp)
        if pack.src_train.shape[0] == len(train_q) and pack.src_val.shape[0] == len(val_q) and pack.src_test.shape[0] == len(test_q):
            return pack
        print('[CACHE] Shape mismatch detected, rebuilding pack.')

    if strategy == 'bow':
        vec = CountVectorizer(max_features=4000, ngram_range=(1, 2))
        src_train = vec.fit_transform(train_q).toarray().astype(np.float32)
        src_val = vec.transform(val_q).toarray().astype(np.float32)
        src_test = vec.transform(test_q).toarray().astype(np.float32)
    elif strategy == 'tfidf':
        vec = TfidfVectorizer(max_features=4000, ngram_range=(1, 2))
        src_train = vec.fit_transform(train_q).toarray().astype(np.float32)
        src_val = vec.transform(val_q).toarray().astype(np.float32)
        src_test = vec.transform(test_q).toarray().astype(np.float32)
    elif strategy == 'bert':
        src_train = mean_pool_transformer_embeddings(train_q, BERT_MODEL_NAME).astype(np.float32)
        src_val = mean_pool_transformer_embeddings(val_q, BERT_MODEL_NAME).astype(np.float32)
        src_test = mean_pool_transformer_embeddings(test_q, BERT_MODEL_NAME).astype(np.float32)
    elif strategy == 'gpt':
        src_train = mean_pool_transformer_embeddings(train_q, GPT_MODEL_NAME).astype(np.float32)
        src_val = mean_pool_transformer_embeddings(val_q, GPT_MODEL_NAME).astype(np.float32)
        src_test = mean_pool_transformer_embeddings(test_q, GPT_MODEL_NAME).astype(np.float32)
    else:
        raise ValueError(f'Unknown dense strategy: {strategy}')

    pack = EmbeddingPack(mode='dense', src_train=src_train, src_val=src_val, src_test=src_test)
    joblib.dump(pack, cp)
    return pack


def build_token_pack(strategy: str, dataset_name: str, train_q: List[str], val_q: List[str], test_q: List[str], train_a: List[str]) -> EmbeddingPack:
    cp = cache_path(dataset_name, strategy, len(train_q), len(val_q), len(test_q))
    if os.path.exists(cp):
        print(f'[CACHE] Loading {strategy} for {dataset_name}')
        pack = joblib.load(cp)
        if pack.src_train.shape[0] == len(train_q) and pack.src_val.shape[0] == len(val_q) and pack.src_test.shape[0] == len(test_q):
            return pack
        print('[CACHE] Shape mismatch detected, rebuilding pack.')

    vocab = Vocabulary(min_freq=MIN_TOKEN_FREQ)
    vocab.build(train_q + train_a)
    src_train = np.array([vocab.encode(x, MAX_SRC_LEN) for x in train_q], dtype=np.int64)
    src_val = np.array([vocab.encode(x, MAX_SRC_LEN) for x in val_q], dtype=np.int64)
    src_test = np.array([vocab.encode(x, MAX_SRC_LEN) for x in test_q], dtype=np.int64)

    pretrained_matrix = None
    if strategy in {'w2v_sg', 'w2v_cbow', 'fasttext'}:
        tokens = [simple_tokenize(x) for x in train_q + train_a]
        vector_size = EMBED_DIM
        if strategy.startswith('w2v'):
            sg = 1 if strategy.endswith('sg') else 0
            emb_model = Word2Vec(sentences=tokens, vector_size=vector_size, window=5, min_count=1, workers=4, sg=sg, seed=SEED)
            kv = emb_model.wv
        else:
            emb_model = FastText(sentences=tokens, vector_size=vector_size, window=5, min_count=1, workers=4, seed=SEED)
            kv = emb_model.wv
        pretrained_matrix = np.random.normal(0, 0.05, (len(vocab.stoi), EMBED_DIM)).astype(np.float32)
        for token, idx in vocab.stoi.items():
            if token in kv:
                pretrained_matrix[idx] = kv[token]
        print(f'[INFO] Built pretrained matrix for {strategy} with shape {pretrained_matrix.shape}')

    pack = EmbeddingPack(mode='token', src_train=src_train, src_val=src_val, src_test=src_test, vocab=vocab, pretrained_matrix=pretrained_matrix)
    joblib.dump(pack, cp)
    return pack


def build_embedding_pack(strategy: str, dataset_name: str, train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame) -> EmbeddingPack:
    train_q = train_df['QuestionText'].tolist()
    val_q = val_df['QuestionText'].tolist()
    test_q = test_df['QuestionText'].tolist()
    train_a = train_df['Answer'].tolist()
    if strategy in {'bow', 'tfidf', 'bert', 'gpt'}:
        return build_dense_pack(strategy, dataset_name, train_q, val_q, test_q)
    return build_token_pack(strategy, dataset_name, train_q, val_q, test_q, train_a)

print('[INFO] Embedding builders ready.')

[INFO] Embedding builders ready.


## Seq2Seq Models and Training

In [ ]:
class QADataset(Dataset):
    def __init__(self, src_data, tgt_ids):
        self.src_data = src_data
        self.tgt_ids = tgt_ids

    def __len__(self):
        return len(self.tgt_ids)

    def __getitem__(self, idx):
        return self.src_data[idx], self.tgt_ids[idx]

def build_target_ids(vocab: Vocabulary, answers: List[str]) -> np.ndarray:
    return np.array([vocab.encode(a, MAX_TGT_LEN) for a in answers], dtype=np.int64)

class Seq2SeqModel(nn.Module):
    def __init__(self, input_mode: str, vocab_size: int, rnn_type: str, bidirectional: bool = False, pretrained_matrix: np.ndarray = None, dense_input_dim: int = None):
        super().__init__()
        self.input_mode = input_mode
        self.rnn_type = rnn_type
        self.bidirectional = bidirectional

        if input_mode == 'token':
            self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
            if pretrained_matrix is not None:
                self.embedding.weight.data.copy_(torch.tensor(pretrained_matrix))
            enc_input_dim = EMBED_DIM
        else:
            if dense_input_dim is None:
                raise ValueError('dense_input_dim is required for dense mode')
            self.dense_projection = nn.Linear(dense_input_dim, EMBED_DIM)
            enc_input_dim = EMBED_DIM

        if rnn_type == 'RNN':
            rnn_cls = nn.RNN
        elif rnn_type == 'LSTM':
            rnn_cls = nn.LSTM
        else:
            rnn_cls = nn.GRU

        self.encoder = rnn_cls(
            input_size=enc_input_dim,
            hidden_size=HIDDEN_DIM,
            num_layers=NUM_LAYERS,
            batch_first=True,
            dropout=DROPOUT if NUM_LAYERS > 1 else 0.0,
            bidirectional=bidirectional,
        )
        enc_out_dim = HIDDEN_DIM * (2 if bidirectional else 1)
        self.decoder = rnn_cls(
            input_size=EMBED_DIM,
            hidden_size=enc_out_dim,
            num_layers=NUM_LAYERS,
            batch_first=True,
            dropout=DROPOUT if NUM_LAYERS > 1 else 0.0,
        )
        self.decoder_emb = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        self.output_head = nn.Linear(enc_out_dim, vocab_size)

    def forward(self, src, tgt_input):
        if self.input_mode == 'token':
            src_emb = self.embedding(src)
        else:
            src_proj = self.dense_projection(src)
            src_emb = src_proj.unsqueeze(1).repeat(1, MAX_SRC_LEN, 1)

        enc_out, enc_state = self.encoder(src_emb)
        dec_emb = self.decoder_emb(tgt_input)

        if isinstance(enc_state, tuple):
            h, c = enc_state
            if self.bidirectional:
                h = torch.cat([h[0::2], h[1::2]], dim=2)
                c = torch.cat([c[0::2], c[1::2]], dim=2)
            dec_state = (h, c)
        else:
            h = enc_state
            if self.bidirectional:
                h = torch.cat([h[0::2], h[1::2]], dim=2)
            dec_state = h

        dec_out, _ = self.decoder(dec_emb, dec_state)
        logits = self.output_head(dec_out)
        return logits

def generate_greedy(model, src_batch, sos_id, eos_id, pad_id):
    model.eval()
    preds = []
    with torch.no_grad():
        batch_size = src_batch.size(0)
        tokens = torch.full((batch_size, 1), sos_id, dtype=torch.long, device=src_batch.device)
        for _ in range(MAX_TGT_LEN - 1):
            logits = model(src_batch, tokens)
            next_tok = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            tokens = torch.cat([tokens, next_tok], dim=1)
        preds = tokens.cpu().numpy().tolist()
    return preds

print('[INFO] Seq2Seq model classes ready.')

[INFO] Seq2Seq model classes ready.


In [8]:
def run_epoch(model, loader, optimizer, criterion, train_mode=True):
    model.train() if train_mode else model.eval()
    losses = []
    loop = tqdm(loader, desc='Train' if train_mode else 'Eval', leave=False)
    for src, tgt in loop:
        if isinstance(src, np.ndarray):
            src = torch.tensor(src)
        if src.dtype in (torch.int32, torch.int64):
            src = src.long().to(DEVICE)
        else:
            src = src.float().to(DEVICE)
        tgt = tgt.long().to(DEVICE)

        dec_in = tgt[:, :-1]
        dec_out = tgt[:, 1:]

        if train_mode:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train_mode):
            logits = model(src, dec_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))
            if train_mode:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        losses.append(float(loss.item()))
        loop.set_postfix(loss=f'{np.mean(losses):.4f}')
    return float(np.mean(losses)) if losses else 0.0

def evaluate_generation(model, loader, vocab: Vocabulary):
    refs, preds = [], []
    sos_id = vocab.stoi[vocab.sos_token]
    eos_id = vocab.stoi[vocab.eos_token]
    pad_id = vocab.stoi[vocab.pad_token]

    for src, tgt in tqdm(loader, desc='Generate', leave=False):
        if isinstance(src, np.ndarray):
            src = torch.tensor(src)
        if src.dtype in (torch.int32, torch.int64):
            src = src.long().to(DEVICE)
        else:
            src = src.float().to(DEVICE)

        pred_ids = generate_greedy(model, src, sos_id, eos_id, pad_id)
        preds.extend([vocab.decode(x) for x in pred_ids])
        refs.extend([vocab.decode(x.tolist()) for x in tgt])

    return compute_bleu(refs, preds), compute_rouge_l(refs, preds), refs, preds

def train_single_configuration(dataset_name: str, embedding_strategy: str, architecture_name: str, train_df, val_df, test_df):
    print('=' * 90)
    print(f'[RUN] Dataset={dataset_name} | Embedding={embedding_strategy} | Architecture={architecture_name}')
    print('=' * 90)

    pack = build_embedding_pack(embedding_strategy, dataset_name, train_df, val_df, test_df)
    if pack.mode == 'dense':
        vocab = Vocabulary(min_freq=1)
        vocab.build(train_df['Answer'].tolist() + val_df['Answer'].tolist() + test_df['Answer'].tolist())
        src_train, src_val, src_test = pack.src_train, pack.src_val, pack.src_test
    else:
        vocab = pack.vocab
        src_train, src_val, src_test = pack.src_train, pack.src_val, pack.src_test

    y_train = build_target_ids(vocab, train_df['Answer'].tolist())
    y_val = build_target_ids(vocab, val_df['Answer'].tolist())
    y_test = build_target_ids(vocab, test_df['Answer'].tolist())

    train_loader = DataLoader(QADataset(src_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(QADataset(src_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(QADataset(src_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

    model = Seq2SeqModel(
        input_mode=pack.mode,
        vocab_size=len(vocab.stoi),
        rnn_type='GRU' if architecture_name == 'BiGRU' else architecture_name,
        bidirectional=(architecture_name == 'BiGRU'),
        pretrained_matrix=pack.pretrained_matrix,
        dense_input_dim=src_train.shape[1] if pack.mode == 'dense' else None,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi[vocab.pad_token])

    history = []
    best_bleu, best_rouge = -1.0, -1.0
    best_path = os.path.join(CKPT_DIR, f'best_{architecture_name}.pt')
    best_payload = None

    for epoch in tqdm(range(1, EPOCHS + 1), desc=f'Epoch loop {architecture_name}', leave=False):
        t0 = time.time()
        train_loss = run_epoch(model, train_loader, optimizer, criterion, train_mode=True)
        val_loss = run_epoch(model, val_loader, optimizer, criterion, train_mode=False)
        val_bleu, val_rouge, _, _ = evaluate_generation(model, val_loader, vocab)
        elapsed = time.time() - t0

        row = {
            'Dataset': dataset_name,
            'EmbeddingStrategy': embedding_strategy,
            'Architecture': architecture_name,
            'Epoch': epoch,
            'TrainLoss': train_loss,
            'ValLoss': val_loss,
            'ValBLEU': val_bleu,
            'ValROUGE_L': val_rouge,
            'RuntimeSec': elapsed,
        }
        history.append(row)
        print(f"[EPOCH {epoch}] train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_bleu={val_bleu:.4f} val_rouge={val_rouge:.4f}")

        better = (val_bleu > best_bleu) or (math.isclose(val_bleu, best_bleu) and val_rouge > best_rouge)
        if better:
            best_bleu, best_rouge = val_bleu, val_rouge
            best_payload = {
                'state_dict': model.state_dict(),
                'vocab': vocab.stoi,
                'dataset_name': dataset_name,
                'embedding_strategy': embedding_strategy,
                'architecture_name': architecture_name,
                'val_bleu': val_bleu,
                'val_rouge_l': val_rouge,
            }
            torch.save(best_payload, best_path)
            print(f'[CHECKPOINT] Updated best {architecture_name} checkpoint at {best_path}')

    if best_payload is None:
        raise RuntimeError('Training did not produce a best checkpoint.')

    model.load_state_dict(best_payload['state_dict'])
    test_bleu, test_rouge, refs, preds = evaluate_generation(model, test_loader, vocab)
    print(f'[TEST] BLEU={test_bleu:.4f} ROUGE-L={test_rouge:.4f}')

    return pd.DataFrame(history), {
        'Dataset': dataset_name,
        'EmbeddingStrategy': embedding_strategy,
        'Architecture': architecture_name,
        'TestBLEU': test_bleu,
        'TestROUGE_L': test_rouge,
        'CheckpointPath': best_path,
        'BestValBLEU': best_bleu,
        'BestValROUGE_L': best_rouge,
    }, pd.DataFrame({'reference': refs, 'prediction': preds})

## Full Experiment Runner, Exports, and Visualizations

In [ ]:
ARCHITECTURES = ['RNN', 'LSTM', 'GRU', 'BiGRU']
EMBEDDING_STRATEGIES = ['bow', 'tfidf', 'w2v_sg', 'w2v_cbow', 'fasttext', 'bert', 'gpt']

RUN_FULL_TRAINING = True  # Set True to run everything
MAX_DATASETS = None        # Optional limiter
MAX_EMBEDDINGS = None      # Optional limiter
MAX_ARCHITECTURES = None   # Optional limiter

all_histories = []
all_test_rows = []
all_examples = []

dataset_items = list(qa_datasets.items())
if MAX_DATASETS is not None:
    dataset_items = dataset_items[:MAX_DATASETS]

embeddings_to_run = EMBEDDING_STRATEGIES if MAX_EMBEDDINGS is None else EMBEDDING_STRATEGIES[:MAX_EMBEDDINGS]
architectures_to_run = ARCHITECTURES if MAX_ARCHITECTURES is None else ARCHITECTURES[:MAX_ARCHITECTURES]

if RUN_FULL_TRAINING:
    for dataset_name, df in tqdm(dataset_items, desc='Datasets'):
        train_df, val_df, test_df = split_dataframe(df)
        print(f'[DATASET] {dataset_name} train={len(train_df)} val={len(val_df)} test={len(test_df)}')
        for emb in tqdm(embeddings_to_run, desc=f'Embeddings for {dataset_name}', leave=False):
            for arch in tqdm(architectures_to_run, desc=f'Architectures for {dataset_name}/{emb}', leave=False):
                try:
                    history_df, test_row, examples_df = train_single_configuration(dataset_name, emb, arch, train_df, val_df, test_df)
                    all_histories.append(history_df)
                    all_test_rows.append(test_row)
                    examples_df['Dataset'] = dataset_name
                    examples_df['EmbeddingStrategy'] = emb
                    examples_df['Architecture'] = arch
                    all_examples.append(examples_df)
                except Exception as exc:
                    print(f'[ERROR] dataset={dataset_name} emb={emb} arch={arch} => {exc}')
else:
    print('[INFO] RUN_FULL_TRAINING is False. Toggle it to start full experiments.')

history_all_df = pd.DataFrame()
test_all_df = pd.DataFrame()
examples_all_df = pd.DataFrame()
best_by_arch = pd.DataFrame()

if all_histories:
    history_all_df = pd.concat(all_histories, ignore_index=True)
    test_all_df = pd.DataFrame(all_test_rows)
    examples_all_df = pd.concat(all_examples, ignore_index=True)

    history_path = os.path.join(OUTPUT_DIR, 'question_answering_seq2seq_all_results.csv')
    best_path = os.path.join(OUTPUT_DIR, 'question_answering_seq2seq_best_models.csv')
    examples_path = os.path.join(OUTPUT_DIR, 'question_answering_seq2seq_examples.csv')

    history_all_df.to_csv(history_path, index=False)
    test_all_df.to_csv(best_path, index=False)
    examples_all_df.to_csv(examples_path, index=False)

    print(f'[SAVE] {history_path}')
    print(f'[SAVE] {best_path}')
    print(f'[SAVE] {examples_path}')

    best_by_arch = (
        test_all_df.sort_values(['Architecture', 'BestValBLEU', 'BestValROUGE_L'], ascending=[True, False, False])
        .groupby('Architecture', as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    manifest = []
    for _, row in best_by_arch.iterrows():
        manifest.append({
            'Architecture': row['Architecture'],
            'BestDataset': row['Dataset'],
            'BestEmbeddingStrategy': row['EmbeddingStrategy'],
            'CheckpointPath': row['CheckpointPath'],
            'BestValBLEU': float(row['BestValBLEU']),
            'BestValROUGE_L': float(row['BestValROUGE_L']),
            'TestBLEU': float(row['TestBLEU']),
            'TestROUGE_L': float(row['TestROUGE_L']),
        })
    manifest_path = os.path.join(OUTPUT_DIR, 'question_answering_seq2seq_best_manifest.json')
    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    print(f'[SAVE] {manifest_path}')

    display(best_by_arch)
else:
    print('[INFO] No training outputs found yet.')

Datasets:   0%|          | 0/7 [00:00<?, ?it/s]

[DATASET] Original_raw train=3831 val=426 test=752


[RUN] Dataset=Original_raw | Embedding=bow | Architecture=RNN


[EPOCH 1] train_loss=8.2768 val_loss=7.8514 val_bleu=0.0079 val_rouge=0.0000
[CHECKPOINT] Updated best RNN checkpoint at .\outputs\qa_seq2sqe_checkpoints\best_RNN.pt


[TEST] BLEU=0.0088 ROUGE-L=0.0000
[RUN] Dataset=Original_raw | Embedding=bow | Architecture=LSTM
[CACHE] Loading bow for Original_raw


...

[TEST] BLEU=0.0077 ROUGE-L=0.0000
[RUN] Dataset=Original_raw | Embedding=fasttext | Architecture=GRU
[CACHE] Loading fasttext for Original_raw


[EPOCH 1] train_loss=8.2902 val_loss=7.8999 val_bleu=0.0027 val_rouge=0.0000
[CHECKPOINT] Updated best GRU checkpoint at .\outputs\qa_seq2sqe_checkpoints\best_GRU.pt


[TEST] BLEU=0.0030 ROUGE-L=0.0000
[RUN] Dataset=Original_raw | Embedding=fasttext | Architecture=BiGRU
[CACHE] Loading fasttext for Original_raw


[ERROR] dataset=Original_raw emb=fasttext arch=BiGRU => Expected hidden size (2, 16, 512), got [2, 16, 256]


[RUN] Dataset=Original_raw | Embedding=bert | Architecture=RNN


## Post-Training Visualizations (Run after training cell)

In [ ]:
if history_all_df.empty:
    print('[INFO] history_all_df is empty. Run the training cell first.')
else:
    print('[INFO] Plotting per-architecture loss and quality curves...')
    for arch in sorted(history_all_df['Architecture'].unique()):
        subset = history_all_df[history_all_df['Architecture'] == arch].copy()
        best_combo = (
            subset.sort_values(['ValBLEU', 'ValROUGE_L'], ascending=False)
            [['Dataset', 'EmbeddingStrategy']].iloc[0]
        )
        sel = subset[(subset['Dataset'] == best_combo['Dataset']) & (subset['EmbeddingStrategy'] == best_combo['EmbeddingStrategy'])]

        plt.figure(figsize=(10, 4))
        plt.plot(sel['Epoch'], sel['TrainLoss'], marker='o', label='TrainLoss')
        plt.plot(sel['Epoch'], sel['ValLoss'], marker='o', label='ValLoss')
        plt.title(f'{arch} Loss Curve | {best_combo["Dataset"]} | {best_combo["EmbeddingStrategy"]}')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(10, 4))
        plt.plot(sel['Epoch'], sel['ValBLEU'], marker='o', label='ValBLEU')
        plt.plot(sel['Epoch'], sel['ValROUGE_L'], marker='o', label='ValROUGE_L')
        plt.title(f'{arch} Quality Curve | {best_combo["Dataset"]} | {best_combo["EmbeddingStrategy"]}')
        plt.xlabel('Epoch')
        plt.ylabel('Score')
        plt.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
if test_all_df.empty:
    print('[INFO] test_all_df is empty. Run the training cell first.')
else:
    print('[INFO] Plotting additional comparison visualizations...')

    # 1) Heatmap of best validation BLEU by architecture and embedding.
    heat_bleu = test_all_df.pivot_table(
        index='Architecture',
        columns='EmbeddingStrategy',
        values='BestValBLEU',
        aggfunc='max'
    )
    plt.figure(figsize=(12, 4))
    sns.heatmap(heat_bleu, annot=True, fmt='.3f', cmap='YlGnBu')
    plt.title('Best Validation BLEU (Architecture x Embedding)')
    plt.tight_layout()
    plt.show()

    # 2) Heatmap of best validation ROUGE-L by architecture and embedding.
    heat_rouge = test_all_df.pivot_table(
        index='Architecture',
        columns='EmbeddingStrategy',
        values='BestValROUGE_L',
        aggfunc='max'
    )
    plt.figure(figsize=(12, 4))
    sns.heatmap(heat_rouge, annot=True, fmt='.3f', cmap='OrRd')
    plt.title('Best Validation ROUGE-L (Architecture x Embedding)')
    plt.tight_layout()
    plt.show()

    # 3) Test BLEU and ROUGE-L grouped by architecture (mean over all combos).
    agg_arch = test_all_df.groupby('Architecture', as_index=False)[['TestBLEU', 'TestROUGE_L']].mean()
    agg_arch_long = agg_arch.melt(id_vars='Architecture', var_name='Metric', value_name='Score')
    plt.figure(figsize=(10, 4))
    sns.barplot(data=agg_arch_long, x='Architecture', y='Score', hue='Metric')
    plt.title('Average Test Metrics by Architecture')
    plt.tight_layout()
    plt.show()

    # 4) Top-10 runs by Test BLEU.
    top10_bleu = test_all_df.sort_values('TestBLEU', ascending=False).head(10).copy()
    top10_bleu['Label'] = top10_bleu['Architecture'] + ' | ' + top10_bleu['EmbeddingStrategy'] + ' | ' + top10_bleu['Dataset']
    plt.figure(figsize=(12, 5))
    sns.barplot(data=top10_bleu, x='TestBLEU', y='Label', color='#4C72B0')
    plt.title('Top 10 Configurations by Test BLEU')
    plt.tight_layout()
    plt.show()